# AT-TPC Latent Space Exploration
---
This notebook explores AT-TPC event embeddings with k-means, PCA projection, PCA variance analysis, UMAP, t-SNE.

In [ ]:
import os
import sys
import numpy as np
sys.path.append('../../')
from clustering import t_SNE_clustering
from clustering import UMAP_embedding
from clustering import k_means_clustering
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, TruncatedSVD

In [ ]:
# create a folder for results of global feature exploration

folder_path = './plots/exploration'
if not os.path.exists(folder_path):
  os.makedirs(folder_path, exist_ok=True)

## 1. Load Data and Configure Preprocessing
---
Set the feature and label paths, then load the `.npy` files.

By default, labels are used exactly as they appear in the file. Plot legends use the label values from the file, such as `0`, `1`, `2` or text values like `cat` and `dog`, unless you provide custom names.

**`label_groups`** (optional): combine raw label values into fewer classes. Set to `None` to skip regrouping. Each inner list becomes one class. Works with numeric or text labels.

**`class_names`** (optional): custom legend names. Leave as `None` to use the label values from the file. If provided, include one name per unique label after any regrouping.

**Preprocessing:** `raw_features` keeps the model's learned embedding geometry. `scaled_features` gives each coordinate equal weight. The notebook uses raw embeddings by default for representation inspection, while allowing optional scaling for distance-based visualizations.

In [ ]:
# Set file paths
features_path = '../../data/features.npy' # CHANGE THE NAME OF THE DATA FILE AS NEEDED
labels_path = '../../data/master_labels.npy' # CHANGE THE NAME OF THE DATA FILE AS NEEDED

# Optional: regroup raw label values into fewer classes.
# None = use labels from the file as-is (default).
label_groups = None
# Numeric example:
# label_groups = [
#     [0, 1, 2],
#     [3],
#     [4, 5],
# ]
# Text example:
# label_groups = [
#     ["cat"],
#     ["dog"],
# ]

# Optional display names for plot legends.
# None = use label values from the file, such as "0", "1", "2" or "cat", "dog".
class_names = None
# class_names = ["Class 0", "Class 1", "Class 2"]

# Preprocessing controls.
# False preserves the model-learned coordinate magnitudes for visualization.
# True gives every embedding coordinate equal weight before distance-based methods.
scale_features_for_visualization = False
preprocess_with_linear_reduction_for_tsne = True #DEFAULT USES PCA TO REDUCE DIMENSIONS
preprocess_with_linear_reduction_for_umap = False
tsne_pca_components = 50
umap_pca_components = 50
random_state = None

raw_features = np.load(features_path)
print(f"Raw features loaded successfully! Shape: {raw_features.shape}")

In [ ]:
def load_labels(label_data):
    labels = np.asarray(label_data).ravel()
    if np.issubdtype(labels.dtype, np.number):
        return labels.astype(int)
    return labels.astype(str)

def apply_label_groups(labels, groups):
    if groups is None:
        return labels
    mapped = labels.copy()
    for new_label, raw_values in enumerate(groups):
        mapped[np.isin(labels, raw_values)] = new_label
    return mapped

def reduce_features_for_manifold(input_features, n_components, method, random_state=None):
    n_components = min(n_components, input_features.shape[0], input_features.shape[1])
    if n_components < 1:
        raise ValueError(f"Cannot reduce features with shape {input_features.shape}.")

    if method == "pca":
        reducer = PCA(n_components=n_components, random_state=random_state)
    elif method == "truncated-svd":
        reducer = TruncatedSVD(n_components=n_components, random_state=random_state)
    else:
        raise ValueError("method must be 'pca' or 'truncated-svd'.")

    reduced_features = reducer.fit_transform(input_features)
    print(f"{method} reduced features to shape: {reduced_features.shape}")
    return reduced_features

# Load labels
label_data = np.load(labels_path)

labels = load_labels(label_data)
labels = apply_label_groups(labels, label_groups)

if class_names is not None and len(class_names) != len(np.unique(labels)):
    raise ValueError(
        f"class_names must have {len(np.unique(labels))} entries, got {len(class_names)}"
    )

if raw_features.ndim != 2:
    raise ValueError(f"raw_features must be a 2D array, got shape {raw_features.shape}")

print("Unique labels:", np.unique(labels))
print("Label counts:", {str(v): int(np.sum(labels == v)) for v in np.unique(labels)})

analysis_features = raw_features
scaled_features = StandardScaler().fit_transform(raw_features)
visualization_features = scaled_features if scale_features_for_visualization else raw_features

print(f"Analysis features shape: {analysis_features.shape}")
print(f"Visualization features shape: {visualization_features.shape}")
print(f"Visualization scaling enabled: {scale_features_for_visualization}")

## 2. K-Means
---
Run k-means on `visualization_features`, separating the embeddings into k clusters.

K-means uses Euclidean distance, so it is sensitive to feature scaling. With `scale_features_for_visualization=False`, clustering preserves model-learned coordinate magnitudes; with `True`, each coordinate contributes equally.

The clusters can then be compared with the grouped labels, using `class_names` for legend text when it is provided.

In [ ]:
save_dir = './plots/k_means'

features_glob, cluster_labels, cluster_indices = k_means_clustering(
    visualization_features,
    labels,
    dimension=2,
    save_dir=save_dir,
    num_samples_to_print=10,
    class_names=class_names,
    random_state=random_state,
)

## 3. PCA Projection
---
Use PCA as a linear baseline before UMAP and t-SNE.

By default this runs on `analysis_features`, which points to `raw_features`, so the projection shows the main directions of variation in the model's actual latent space. If you intentionally want a correlation-style PCA where each coordinate has equal weight, set `analysis_features = scaled_features` before running this cell.

PCA can project the embeddings into either 2D or 3D by changing `dimension`, and uses the same labels and optional `class_names` as the other plots.

In [ ]:
from clustering import pca_clustering

save_dir = './plots/pca'

# Use dimension=2 for a 2D plot or dimension=3 for a 3D plot.
pca_results = pca_clustering(
    analysis_features,
    labels,
    dimension=2,
    save_dir=save_dir,
    class_names=class_names,
    random_state=random_state,
)

## 3b. PCA Variance Analysis
---
Fit PCA on `analysis_features` and report how many principal components are needed to reach a target explained-variance threshold (default 95%).

With the default `analysis_features = raw_features`, this measures how variance is distributed in the learned latent space. If `analysis_features` is changed to `scaled_features`, the result answers the equal-coordinate-weight version of the same question.

The cumulative variance plot is saved under `./plots/pca/variance/`.

In [ ]:
from clustering import pca_variance_analysis

variance_threshold = 0.95

variance_results = pca_variance_analysis(
    analysis_features,
    variance_threshold=variance_threshold,
    save_dir="./plots/pca",
    plot_name="exploration",
    random_state=random_state,
)

variance_results["n_components"]

## 4. Check Cluster Labels
---
Print the true labels for the sampled events in each cluster.

In [ ]:
def true_labels(indices, labels):
    c_labels = []
    for index in indices:
        c_labels.append(labels[index])

    return c_labels

print("True labels verification per cluster:")
for i, cluster in enumerate(cluster_indices):
    print(f"Cluster {i}: {true_labels(cluster, labels)}")

## 5. UMAP Visualization
---
Run UMAP on `visualization_features` and color the 2D plot by label.
 
UMAP is distance-based, so scaling changes the neighborhood graph: keep `scale_features_for_visualization=False` to preserve learned embedding magnitudes, or set it to `True` to give each coordinate equal weight. For very high-dimensional (1000+ dimensions) or noisy inputs, set `preprocess_with_linear_reduction_for_umap=True` to apply PCA first for dense embeddings. 

The main hyperparameter to change is `neighbors`, which controls how much local structure UMAP preserves. The plot uses label values from the file unless `class_names` is set, saves under `./plots/umap/`, and stores the coordinates in `umap_results`.

In [ ]:
neighbors = 50
umap_input_features = visualization_features
if preprocess_with_linear_reduction_for_umap:
    umap_input_features = reduce_features_for_manifold(
        visualization_features,
        n_components=umap_pca_components,
        method='pca',
        random_state=random_state,
    )

fig, ax = plt.subplots(figsize=(10, 7))

umap_results = UMAP_embedding(
    umap_input_features,
    dimension=2,
    ax=ax,
    labels=labels,
    neighbors=neighbors,
    class_names=class_names,
    save_dir='./plots',
    random_state=random_state,
)

ax.legend()
plt.title(f"UMAP 2D Manifold Embedding (n_neighbors={neighbors})")
plt.savefig('./plots/umap/umap_unified_manifold.png', dpi=200)
plt.show()

## 6. t-SNE Visualization
---
Run t-SNE on a linearly reduced version of `visualization_features` and color the 2D plot by label. The t-SNE paper and scikit-learn documentation recommend reducing high-dimensional inputs to a reasonable intermediate size before t-SNE, commonly around 50 dimensions. This notebook uses PCA by default for dense latent embedding arrays. 

Scaling remains controlled by `scale_features_for_visualization`, because equal-coordinate weighting may or may not be appropriate for learned embeddings.

The main hyperparameter to change is `perplexity`, which controls how many nearby points t-SNE considers when building the embedding. t-SNE is useful for local structure, but far-apart distances should be interpreted carefully. The plot is saved under `./plots/t-sne/`, and the coordinates are stored in `tsne_results`.

In [ ]:
perplexity = 40
tsne_input_features = visualization_features
if preprocess_with_linear_reduction_for_tsne:
    tsne_input_features = reduce_features_for_manifold(
        visualization_features,
        n_components=tsne_pca_components,
        method='pca',
        random_state=random_state,
    )

fig, ax = plt.subplots(figsize=(10, 7))

tsne_results = t_SNE_clustering(
    tsne_input_features,
    dimension=2,
    ax=ax,
    labels=labels,
    perplexity=perplexity,
    class_names=class_names,
    save_dir='./plots',
    random_state=random_state,
)

ax.legend()
plt.title(f"t-SNE 2D Manifold Embedding (perplexity={perplexity})")
plt.savefig('./plots/t-sne/tsne_unified_manifold.png', dpi=200)
plt.show()